# Fill jawaban/cara kosong — DeepSeek-R1-Distill-Qwen-7B
### Versi `transformers` (anti error flashinfer / -lcuda)

Tanpa vllm. Lebih lambat tapi stabil. **Resumable**: sesi putus -> Run All lagi, lanjut otomatis.

**Setting panel kanan:** GPU **T4 x2**, Internet **On**, Add Input = `dataset_dedup.jsonl`.

### 1. Install

In [ ]:
!pip install -q -U transformers accelerate

### 2. Konfigurasi (path data dicari otomatis)

In [ ]:
import json, glob, os, torch
hits = glob.glob('/kaggle/input/**/dataset_dedup.jsonl', recursive=True)
INPUT = hits[0] if hits else '/kaggle/input/dataset_dedup.jsonl'
CACHE = '/kaggle/working/fill_cache.jsonl'
OUTPUT = '/kaggle/working/dataset_filled.jsonl'
MODEL_ID = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
MAX_NEW_TOKENS = 2048   # naikin kalau cara sering kepotong; turunin biar cepat
BATCH_SIZE = 8          # turunin ke 4 kalau OOM
print('INPUT =', INPUT)
print('GPU   :', torch.cuda.device_count(), 'x', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

### 3. Helper (inline)

In [ ]:
BS = chr(92); NL = chr(10)

def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def extract_boxed(text):
    if not text:
        return None
    marker = BS + 'boxed'
    idx = text.rfind(marker)
    if idx == -1:
        return None
    i = idx + len(marker)
    while i < len(text) and text[i] != '{':
        if not text[i].isspace():
            return None
        i += 1
    if i >= len(text):
        return None
    depth = 0; out = []
    for ch in text[i:]:
        if ch == '{':
            depth += 1
            if depth == 1:
                continue
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return ''.join(out)
        out.append(ch)
    return None

def needs_fill(r):
    return not (r.get('jawaban') or '').strip() or not (r.get('cara') or '').strip()

def make_prompt(soal):
    head = ('Selesaikan soal matematika berikut. Tunjukkan langkah-langkah '
            'penyelesaian secara rinci dalam Bahasa Indonesia. Pastikan jawaban '
            'akhir berada di dalam ')
    return head + BS + 'boxed{}.' + NL + NL + soal

### 4. Load data + resume

In [ ]:
rows = load_jsonl(INPUT)
done = {}
if os.path.exists(CACHE):
    for r in load_jsonl(CACHE):
        done[r['soal']] = r
todo = [r for r in rows if needs_fill(r) and r['soal'] not in done]
print('total', len(rows), '| perlu fill', sum(needs_fill(r) for r in rows),
      '| sudah di cache', len(done), '| sisa diproses', len(todo))

### 5. Load model (transformers, dibagi ke 2 GPU)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = 'left'                 # wajib utk batched generation
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',                    # sebar ke 2x T4
    attn_implementation='sdpa',           # native PyTorch, tanpa flashinfer
)
model.eval()
print('model loaded ->', model.hf_device_map if hasattr(model,'hf_device_map') else 'ok')

### 6. Generate (batched, resumable)

In [ ]:
import time
t0 = time.time()
with open(CACHE, 'a', encoding='utf-8') as cf:
    for i in range(0, len(todo), BATCH_SIZE):
        batch = todo[i:i+BATCH_SIZE]
        prompts = [tok.apply_chat_template(
            [{'role':'user','content':make_prompt(r['soal'])}],
            tokenize=False, add_generation_prompt=True) for r in batch]
        enc = tok(prompts, return_tensors='pt', padding=True,
                  truncation=True, max_length=2048).to(0)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                                 do_sample=True, temperature=0.6, top_p=0.95,
                                 pad_token_id=tok.pad_token_id)
        gens = tok.batch_decode(out[:, enc.input_ids.shape[1]:],
                                skip_special_tokens=True)
        for r, gen in zip(batch, gens):
            boxed = extract_boxed(gen)
            jaw = (r.get('jawaban') or '').strip()
            filled = {'soal': r['soal'], 'cara': gen.strip(),
                      'jawaban': jaw if jaw else (boxed or ''),
                      '_fill_ok': bool(jaw) or boxed is not None}
            done[r['soal']] = filled
            cf.write(json.dumps(filled, ensure_ascii=False) + NL)
            cf.flush()
        n = min(i+BATCH_SIZE, len(todo))
        el = time.time()-t0
        print(f'progress {n}/{len(todo)} | {el/n:.1f}s/soal | est sisa {el/n*(len(todo)-n)/60:.0f} mnt')

### 7. Gabung -> output final + statistik

In [ ]:
nfail = 0
with open(OUTPUT, 'w', encoding='utf-8') as out:
    for r in rows:
        f = done.get(r['soal'])
        if f is not None:
            if not f.get('_fill_ok', True):
                nfail += 1
            rec = {'soal': f['soal'], 'cara': f['cara'], 'jawaban': f['jawaban']}
        else:
            rec = {'soal': r['soal'], 'cara': r.get('cara',''), 'jawaban': r.get('jawaban','')}
        out.write(json.dumps(rec, ensure_ascii=False) + NL)
still_empty = sum(1 for r in load_jsonl(OUTPUT) if not (r.get('jawaban') or '').strip())
print('DONE ->', OUTPUT)
print('fill gagal (no boxed):', nfail, '| jawaban masih kosong:', still_empty)
print('Download dataset_filled.jsonl dari panel Output (kanan).')